#### Import data

In [1]:
from applied.data_processing import (
    load_operating_data,
    load_product_data,
    build_features_and_target,
)

In [2]:
from pathlib import Path

# Project root = one level above notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

operating_csv = DATA_DIR / "4000 series operating data.csv"
product_xlsx = DATA_DIR / "4000 series product data.xlsx"

In [3]:
op_df = load_operating_data(operating_csv)
op_df.head()

,Date and time,Batch,LIQUID,LIQUID.1,LIQUID.2,LIQUID.3,LIQUID.4,LIQUID.5,pH,GAS,GAS.1,GAS.2,GAS.3,OFFGAS,OFFGAS.1,PRESSURE,PRESSURE.1,OXYGEN
2,2019-04-02 08:46:00,4030,1049.57,25.91,14.44,NaN,297.73,14980.0,5.76,NaN,NaN,NaN,55.14,1.87,16.94,1.79,5.15,6.78
3,2019-04-02 09:01:00,4030,1049.09,25.62,13.54,NaN,357.44,15000.0,5.79,NaN,NaN,NaN,56.88,1.89,20.52,1.80,5.16,8.39
4,2019-04-02 09:16:00,4030,1049.61,25.44,13.59,NaN,356.83,15010.0,5.80,NaN,NaN,NaN,56.03,1.94,23.77,1.80,5.15,8.07
5,2019-04-02 09:31:00,4030,1047.57,25.59,13.97,NaN,356.77,15000.0,5.79,NaN,NaN,NaN,53.91,2.00,27.01,1.80,5.15,7.23
6,2019-04-02 09:46:00,4030,1048.16,25.49,13.43,NaN,357.21,15010.0,5.78,NaN,NaN,NaN,53.97,2.05,30.15,1.80,5.13,7.16


In [4]:
op_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83204 entries, 2 to 83205
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date and time  35696 non-null  datetime64[us]
 1   Batch          83204 non-null  int64         
 2   LIQUID         82777 non-null  float64       
 3   LIQUID.1       82786 non-null  float64       
 4   LIQUID.2       82781 non-null  float64       
 5   LIQUID.3       80023 non-null  float64       
 6   LIQUID.4       82782 non-null  float64       
 7   LIQUID.5       82806 non-null  float64       
 8   pH             82776 non-null  float64       
 9   GAS            82068 non-null  float64       
 10  GAS.1          77992 non-null  float64       
 11  GAS.2          76913 non-null  float64       
 12  GAS.3          82779 non-null  float64       
 13  OFFGAS         82765 non-null  float64       
 14  OFFGAS.1       82774 non-null  float64       
 15  PRESSURE       82784 non-null 

In [5]:
op_df.filter(like="LIQUID").describe()

,LIQUID,LIQUID.1,LIQUID.2,LIQUID.3,LIQUID.4,LIQUID.5
count,82777.000000,82786.000000,82781.000000,80023.000000,82782.000000,82806.000000
mean,1163.627119,34.676143,18.860966,43.498032,493.128547,22780.386808
std,232.743444,4.488459,2.425335,8.946294,66.943492,2022.294729
min,0.000000,0.020000,0.000000,0.000000,0.000000,0.000000
25%,1109.860000,32.770000,18.070000,41.830000,467.750000,22500.000000
50%,1210.300000,36.370000,19.550000,46.940000,516.100000,23050.000000
75%,1310.070000,37.230000,20.220000,47.030000,532.837500,23530.000000
max,11375.000000,53.220000,65.410000,169.970000,784.800000,44240.000000


In [6]:
op_df.filter(like="LIQUID").corr()

,LIQUID,LIQUID.1,LIQUID.2,LIQUID.3,LIQUID.4,LIQUID.5
LIQUID,1.000000,0.483750,0.472656,0.261016,0.492351,0.137088
LIQUID.1,0.483750,1.000000,0.926064,0.520520,0.921240,0.162741
LIQUID.2,0.472656,0.926064,1.000000,0.505958,0.885237,0.178432
LIQUID.3,0.261016,0.520520,0.505958,1.000000,0.546279,0.146249
LIQUID.4,0.492351,0.921240,0.885237,0.546279,1.000000,0.169579
LIQUID.5,0.137088,0.162741,0.178432,0.146249,0.169579,1.000000


In [11]:
op_df.isna().sum()

Date and time    47508
Batch                0
LIQUID             427
LIQUID.1           418
LIQUID.2           423
LIQUID.3          3181
LIQUID.4           422
LIQUID.5           398
pH                 428
GAS               1136
GAS.1             5212
GAS.2             6291
GAS.3              425
OFFGAS             439
OFFGAS.1           430
PRESSURE           420
PRESSURE.1         423
OXYGEN             424
dtype: int64

In [15]:
groups = op_df.groupby("Batch", sort=False)

In [18]:
batch_ids = groups.groups.keys()
batch_ids

dict_keys([4030, 4032, 4033, 4034, 4035, 4036, 4037, 4038, 4039, 4040, 4041, 4042, 4043, 4044, 4045, 4046, 4047, 4048, 4050, 4051, 4052, 4053])

In [23]:
batch_df = groups.get_group(4030)

batch_df.head()

,Date and time,Batch,LIQUID,LIQUID.1,LIQUID.2,LIQUID.3,LIQUID.4,LIQUID.5,pH,GAS,GAS.1,GAS.2,GAS.3,OFFGAS,OFFGAS.1,PRESSURE,PRESSURE.1,OXYGEN
2,2019-04-02 08:46:00,4030,1049.57,25.91,14.44,NaN,297.73,14980.0,5.76,NaN,NaN,NaN,55.14,1.87,16.94,1.79,5.15,6.78
3,2019-04-02 09:01:00,4030,1049.09,25.62,13.54,NaN,357.44,15000.0,5.79,NaN,NaN,NaN,56.88,1.89,20.52,1.80,5.16,8.39
4,2019-04-02 09:16:00,4030,1049.61,25.44,13.59,NaN,356.83,15010.0,5.80,NaN,NaN,NaN,56.03,1.94,23.77,1.80,5.15,8.07
5,2019-04-02 09:31:00,4030,1047.57,25.59,13.97,NaN,356.77,15000.0,5.79,NaN,NaN,NaN,53.91,2.00,27.01,1.80,5.15,7.23
6,2019-04-02 09:46:00,4030,1048.16,25.49,13.43,NaN,357.21,15010.0,5.78,NaN,NaN,NaN,53.97,2.05,30.15,1.80,5.13,7.16


In [9]:
prod_df = load_product_data(product_xlsx)
prod_df.head()

,Date and time,Batch,Product
2,2019-02-04 00:00:00,4030,5.9
3,2019-02-04 02:00:00,4030,8.2
4,2019-02-04 04:00:00,4030,9.7
5,2019-02-04 06:00:00,4030,14.3
6,2019-02-04 08:00:00,4030,16.4


### Aggregating Parallel Sensor Channels

In the operating data, several process variables (e.g. liquid inflow,
gas inflow, pressure) are measured using multiple parallel sensor
channels. These channels correspond to different physical streams or
sensors measuring the same underlying quantity, rather than distinct
process variables.

For example, the liquid inflow is recorded across six parallel channels
(`LIQUID_1`–`LIQUID_6`). Since all channels represent contributions to the
same physical inflow, it is appropriate to aggregate them into a single
total liquid inflow variable.

The aggregation is performed **at each measurement timestamp**. The
total liquid inflow at time $t$ is defined as

$$
\text{LIQUID}_{\text{total}}(t)
=
\sum_{i=1}^{6} \text{LIQUID}_i(t)
$$

If one or more channels are missing at a given time, the total inflow is
computed as the sum of the available channels only. Missing channel
values are not treated as zero, which avoids artificially inflating or
deflating the estimated inflow.

This aggregation is physically meaningful, reduces redundancy and
multicollinearity in the feature space, and aligns with the coursework
specification, which defines the product rate using the *total* liquid
inflow rather than individual streams.

A similar aggregation strategy is applied to other parallel sensor
groups (e.g. gas inflow and pressure), using summation or averaging as
appropriate based on the physical interpretation of each variable.


In [9]:
# --- Create totals ---

op_df["TOTAL_LIQUID_INFLOW"] = op_df[
    ["LIQUID", "LIQUID.1", "LIQUID.2",
     "LIQUID.3", "LIQUID.4", "LIQUID.5"]
].sum(axis=1)

op_df["TOTAL_GAS_INFLOW"] = op_df[
    ["GAS", "GAS.1", "GAS.2", "GAS.3"]
].sum(axis=1)

op_df["TOTAL_OFFGAS"] = op_df[
    ["OFFGAS", "OFFGAS.1"]
].mean(axis=1)   # mean is better than sum for sensors

op_df["MEAN_PRESSURE"] = op_df[
    ["PRESSURE", "PRESSURE.1"]
].mean(axis=1)


# --- Drop original columns ---

op_df = op_df.drop(columns=[
    "LIQUID", "LIQUID.1", "LIQUID.2",
    "LIQUID.3", "LIQUID.4", "LIQUID.5",
    "GAS", "GAS.1", "GAS.2", "GAS.3",
    "OFFGAS", "OFFGAS.1",
    "PRESSURE", "PRESSURE.1"
])

In [10]:
op_df

,Date and time,Batch,pH,OXYGEN,TOTAL_LIQUID_INFLOW,TOTAL_GAS_INFLOW,TOTAL_OFFGAS,MEAN_PRESSURE
2,2019-04-02 08:46:00,4030,5.76,6.78,16367.65,55.14,9.405,3.470
3,2019-04-02 09:01:00,4030,5.79,8.39,16445.69,56.88,11.205,3.480
4,2019-04-02 09:16:00,4030,5.80,8.07,16455.47,56.03,12.855,3.475
5,2019-04-02 09:31:00,4030,5.79,7.23,16443.90,53.91,14.505,3.475
6,2019-04-02 09:46:00,4030,5.78,7.16,16454.29,53.97,16.100,3.465
...,...,...,...,...,...,...,...,...
83201,NaT,4053,5.83,113.78,13799.79,11741.07,2.590,3.555
83202,NaT,4053,5.72,114.99,21098.71,11347.51,2.555,3.540
83203,NaT,4053,5.62,115.00,21840.41,10896.03,2.570,3.530
83204,NaT,4053,5.53,115.00,21838.40,10562.03,2.555,3.520
